In [10]:
%load_ext autoreload
%autoreload 2
from IPython.core.display import display, HTML
display(HTML("<style>.container { width:95% !important; }</style>"))
import warnings
warnings.filterwarnings("ignore")

import os
import pandas as pd
import sys
sys.path.insert(0, '/home/kat/Repos/SALSA/')

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


### Load in the cleaned HIV dataset.

In [11]:
tokens_salsa = '#%()+-0123456789<=>BCFHILNOPRSX[]cnos'
tokens_hiv = 'Ub)gsP5u=8H[d]noW06%2GLlO+NFSTa9ZCeI-r4(1ihAVBRc3M7pt#'

df_hiv = pd.read_csv('data/model_ready/hiv/train/anchor_smiles.csv')
hiv_vocab = list(set(list(''.join(df_hiv.smiles.values))))
print(len(hiv_vocab))
print(len(tokens_salsa))

34
37


### Load models and get latents ... 

In [15]:
from benchmark_utils import get_latents

In [ ]:
tags = ['2022041804_04', # salsa
        '2022041807_a03', # contrastive  
        '2022041809_a04', # vanilla ae
       ]

In [16]:
# # # # # # # # # # # #
tag = '2022041804_04' 
load_bs = 32
# # # # # # # # # # # #

X_train = get_latents(tag, 'train', load_bs) 
X_test = get_latents(tag, 'test', load_bs) 

Using 4 GPUs!
Loaded model weights from /home/kat/Repos/SALSA/results/models/2022041804_04/29.pt!


995it [02:18,  7.16it/s]                         


Using 4 GPUs!
Loaded model weights from /home/kat/Repos/SALSA/results/models/2022041804_04/29.pt!


119it [00:16,  7.12it/s]                         


In [17]:
y_train = pd.read_csv('data/model_ready/hiv/train/labels.csv')['y'].values
y_test = pd.read_csv('data/model_ready/hiv/test/labels.csv')['y'].values

print(X_train.shape, y_train.shape)
print(X_test.shape, y_test.shape)

# for holding results ... 
df_y = pd.DataFrame(y_test, columns=['y_test'])

(31829, 32) (31829,)
(3782, 32) (3782,)


### Run single random forest ... 

In [24]:
from imblearn.ensemble import BalancedRandomForestClassifier as BalRF
from sklearn.model_selection import StratifiedKFold

seedy = 666

clf = BalRF(n_estimators=100, max_features="auto",sampling_strategy='auto',
            random_state=seedy,n_jobs=-1,oob_score=True)

clf.fit(X_train,y_train)

y_pred = clf.predict(X_test)
y_prob = clf.predict_proba(X_test)

df_y['y_prob'] = y_prob[:,1]
df_y['y_pred'] = y_pred

In [25]:
df_y

,y_test,y_prob,y_pred
0,0.0,0.45,0.0
1,0.0,0.47,0.0
2,0.0,0.40,0.0
3,0.0,0.46,0.0
4,0.0,0.47,0.0
...,...,...,...
3777,0.0,0.43,0.0
3778,0.0,0.36,0.0
3779,0.0,0.41,0.0
3780,0.0,0.31,0.0


In [26]:
from benchmark_utils import get_metrics
get_metrics(df_y.y_test, df_y.y_pred, df_y.y_prob)

{'Sn': 0.4666666666666667,
 'Sp': 0.7974207415368082,
 'Pr': 0.03580562659846547,
 'BAcc': 0.6320437041017375,
 'Average Precision': 0.025170424088978932,
 'AUROC': 0.7019657890023284}

In [47]:
# from contra_seq_dataset import *
# from torch.utils.data import DataLoader, RandomSampler

# p = 'data/model_ready/hiv/train/anchor_smiles.csv'

# ds = ContraSeqDataset(p)
# df = get_dataset_array(p)

# bs = 32

# from tqdm.notebook import tqdm
# loader = DataLoader(ds, batch_size=bs, sampler=range(len(df)), 
#                     num_workers=0, pin_memory=True)
# latents = []
# for samp in tqdm(loader, total=len(df)//bs):
#     for k,v in samp.items():
#         if torch.is_tensor(v):
#             samp[k] = v.to(device)
#     latent, _ = model.forward(samp['seq'], samp['pad_mask'], 
#                               samp['avg_mask'], samp['out_mask'])
#     latent = latent.cpu().detach().numpy()
#     latents.append(latent)
# latents = np.concatenate(latents, axis=0)

In [ ]:
# from imblearn.ensemble import BalancedRandomForestClassifier as BalRF
# from sklearn.model_selection import StratifiedKFold

# seedy = 666

# skf = StratifiedKFold(n_splits=5)
# skf.get_n_splits(X, y)

# clf = BalRF(n_estimators=100, max_features="auto",sampling_strategy='auto',
#             random_state=seedy,n_jobs=-1,oob_score=True)

# df_y = pd.DataFrame(y_binary, columns=['y_test'])

# oobs = []
# for i,(train_idx, test_idx) in enumerate(skf.split(X,y)):
#     X_train, X_test = X[train_idx], X[test_idx]
#     y_train, y_test = y[train_idx], y[test_idx]

#     clf.fit(X_train,y_train)
#     print(f'OOB, split {i}: {clf.oob_score_}')

#     y_pred = clf.predict(X_test)
#     y_prob = clf.predict_proba(X_test)
#     df_y.loc[test_idx,'y_prob'] = y_prob[:,1]
#     df_y.loc[test_idx,'y_pred'] = y_pred
#     oobs.append(clf.oob_score_)
    
# print(f"Mean OOB: {sum(oobs)/5}")